## Подготовка данных для моделей

In [2]:
import pandas as pd
import numpy as np

# выгрузим таблицу с данными пресс-релизов
press_df = pd.read_json('../EDA/cbr_key-rate_press_releases_processed.json')
# выгрузим таблицу с ключевой ставкой
key_rate_df = pd.read_csv('../keyRateForecast-Polina 2/data/labels/key_rate_history.csv')
# выгрузим таблицу с инфляцией
inflation_df = pd.read_csv('../Parsing/key_inflation.csv')
# выгрузим таблицу с курсом доллара
currency_df = pd.read_csv('../Parsing/currency_rate.csv')
# выгрузим таблицу с будущим решением
timeline = pd.read_csv("../keyRateForecast-Polina 2/data/processed/dataset_timeline.csv")

In [3]:
# Посмотрим, как называются колонки
print(press_df.columns)
print(key_rate_df.columns)
print(inflation_df.columns)
print(currency_df.columns)

Index(['date', 'title', 'text', 'url', 'text_clean', 'text_sentences',
       'text_tokens', 'text_lemmas', 'text_bigrams', 'title_clean',
       'title_sentences', 'title_tokens', 'title_lemmas', 'title_bigrams',
       'decision', 'key_rate'],
      dtype='object')
Index(['date', 'rate', 'rate_next', 'label'], dtype='object')
Index(['date', 'key-rate', 'inflation'], dtype='object')
Index(['date', 'dollar_rate'], dtype='object')


In [4]:
# приводим даты в каждой таблицы к формату datetime
import dateparser
press_df["date"] = press_df["date"].apply(lambda x: dateparser.parse(x))
key_rate_df['date'] = pd.to_datetime(key_rate_df['date'])
inflation_df['date'] = inflation_df['date'].astype(str)
n = 13
for i in range(57, 69):
    n -= 1
    inflation_df.loc[i, 'date'] = f'{n}.2020'
inflation_df["date"] = pd.to_datetime(inflation_df["date"], format="%m.%Y")
currency_df['date'] = pd.to_datetime(currency_df['date'], format="%d.%m.%Y")

/var/folders/lp/20xw8zx12gq7xct54lgfpkyw0000gn/T/ipykernel_61240/2968020334.py:3: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  press_df["date"] = press_df["date"].apply(lambda x: dateparser.parse(x))


In [5]:
# очистим от пресс-релизов по которым не выявлены
print(f'Количество строк пресс-релизов до очистики {press_df.shape}')
press_df.dropna(subset=['decision'], inplace=True)
press_df.reset_index(drop=True, inplace=True)
print(f'Количество строк пресс-релизов после очистики {press_df.shape}')

Количество строк пресс-релизов до очистики (108, 16)
Количество строк пресс-релизов после очистики (103, 16)


In [6]:
# приведем инфояцию к числовому формату
inflation_df['inflation'] = (inflation_df['inflation'].str.replace(',', '.', regex=False))
inflation_df['inflation'] = pd.to_numeric(inflation_df['inflation'], errors='coerce')

# делаем лаги
inflation_df = inflation_df.sort_values('date').copy()
inflation_df['inflation_lag_1'] = inflation_df['inflation'].shift(1)
inflation_df['inflation_lag_2'] = inflation_df['inflation'].shift(2)
inflation_df['inflation_lag_3'] = inflation_df['inflation'].shift(3)

# добавляем разницу между инфляцией и целью по инфляции
TARGET_INFLATION = 4.0
inflation_df['inflation_gap'] = inflation_df['inflation'] - TARGET_INFLATION

In [7]:
# сортируем по дате
key_rate_df = key_rate_df.sort_values('date').copy()
# добавляем предыдущую ставку 
key_rate_df['key_rate'] = key_rate_df['rate']
key_rate_df['key_rate_prev'] = key_rate_df['key_rate'].shift(1)
# добавляем разницу между ставками
key_rate_df['key_rate_change'] = key_rate_df['key_rate'] - key_rate_df['key_rate_prev']

In [8]:
# сортируем курс доллара по дате
currency_df = currency_df.sort_values('date').copy()
# переводим курс в числовой тип данных
currency_df['usd_rate'] = (currency_df['dollar_rate'].str.replace(',', '.', regex=False))
currency_df['usd_rate'] = pd.to_numeric(currency_df['usd_rate'], errors='coerce')

In [9]:
# добавляем лаги по курсу доллара
currency_df['usd_lag_1'] = currency_df['usd_rate'].shift(1)
currency_df['usd_lag_2'] = currency_df['usd_rate'].shift(2)
currency_df['usd_lag_3'] = currency_df['usd_rate'].shift(3)

In [10]:
# строим единый датасет

base_df = press_df[['date', 'decision', 'title_clean', 'text_clean', 'text_tokens', 'url']].copy()
base_df = base_df.sort_values('date').reset_index(drop=True)

In [11]:
# добавляем к текстам данные по ставке
df = base_df.merge(
    key_rate_df[['date', 'key_rate', 'key_rate_prev', 'key_rate_change']],
    on='date',
    how='left'
)
print("[merge] + key_rate:", df.shape)

[merge] + key_rate: (103, 9)


In [12]:
# добавляем к текстам данные инфляции
df = df.sort_values('date')
inflation_df = inflation_df.sort_values('date')

df = pd.merge_asof(
    df,
    inflation_df[['date', 'inflation', 'inflation_lag_1',
                  'inflation_lag_2', 'inflation_lag_3', 'inflation_gap']],
    on='date',
    direction='backward'
)
print("[merge] + inflation:", df.shape)

[merge] + inflation: (103, 14)


In [13]:
# добавляем к текстам данные по курсу доллара
df = df.merge(
    currency_df[['date', 'usd_rate', 'usd_lag_1', 'usd_lag_2', 'usd_lag_3']],
    on='date',
    how='left'
)
print("[merge] + usd:", df.shape)

[merge] + usd: (103, 18)


In [14]:
# очистим от строк, где нет курса доллара
print(f'Количество строк пресс-релизов до очистики {df.shape}')
df.dropna(subset=['usd_rate'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Количество строк пресс-релизов после очистики {df.shape}')

Количество строк пресс-релизов до очистики (103, 18)
Количество строк пресс-релизов после очистики (98, 18)


In [15]:
# перемименуем решение, так как это решение текущее, а не будущее
df = df.rename(columns={"decision": "decision_text"})

In [16]:
# из таймлайна сделаем данные для решений
timeline = timeline[["url", "decision"]].dropna(subset=["decision"])
timeline["decision"] = timeline["decision"].astype(int)
timeline = timeline.drop_duplicates(subset=["url"])
timeline = timeline.rename(columns={"decision": "decision_future"})

In [17]:
# сопоставим данные
df = df.merge(timeline, on="url", how="left")

In [18]:
set_df = set(df["url"].dropna().unique())
set_timeline = set(timeline["url"].dropna().unique())

print("В df есть URL, которых нет в timeline:", len(set_df - set_timeline))
print("В timeline есть URL, которых нет в df:", len(set_timeline - set_df))


В df есть URL, которых нет в timeline: 44
В timeline есть URL, которых нет в df: 116


In [19]:
print("Строк без таргета:", df["decision_future"].isna().sum())

Строк без таргета: 44


Добавим в получившийся датасет признаки, которые связаны с анализом текста в EDA.

In [20]:
# длина в символах
df["text_len_chars"] = df["text_clean"].astype(str).str.len()

# длина в токенах
df["text_len_tokens"] = df["text_tokens"].apply(len)

In [21]:
# количество уникальных слов
df["text_unique_tokens"] = df["text_tokens"].apply(lambda x: len(set(x)))

# доля уникальных слов
df["text_unique_ratio"] = df["text_unique_tokens"] / (df["text_len_tokens"] + 1)

In [ ]:
from pathlib import Path
import pandas as pd

# частотный словарь только для HOLD
hold_df = pd.read_csv(
    '../keyRateForecast-Polina 2/reports/top_words_before_hold.csv'
)

# log-odds словари для HIKE и CUT
log_hike = pd.read_csv(
    '../keyRateForecast-Polina 2/reports/logodds_top_hike.csv'
)
log_cut  = pd.read_csv(
    '../keyRateForecast-Polina 2/reports/logodds_top_cut.csv'
)

TOP_N = 50

# словарь слов для повышения/понижения по log-odds
vocab_hike = set(
    log_hike.sort_values("log_odds", ascending=False)
            .head(TOP_N)["word"]
)
vocab_cut = set(
    log_cut.sort_values("log_odds", ascending=True)
           .head(TOP_N)["word"]
)

# словарь слов для сохранения ставки по простым топам
vocab_hold = set(
    hold_df.sort_values("count", ascending=False)
           .head(TOP_N)["word"]
)

print("слов для hike (log-odds):", len(vocab_hike))
print("слов для cut  (log-odds):", len(vocab_cut))
print("слов для hold (freq):    ", len(vocab_hold))

def count_from_vocab(tokens, vocab):
    if not isinstance(tokens, list):
        return 0
    return sum(1 for t in tokens if t in vocab)

if "text_len_tokens" not in df.columns:
    df["text_len_tokens"] = df["text_tokens"].apply(len)

df["hike_words_count"] = df["text_tokens"].apply(
    lambda toks: count_from_vocab(toks, vocab_hike)
)
df["cut_words_count"] = df["text_tokens"].apply(
    lambda toks: count_from_vocab(toks, vocab_cut)
)
df["hold_words_count"] = df["text_tokens"].apply(
    lambda toks: count_from_vocab(toks, vocab_hold)
)

df["hike_words_ratio"] = df["hike_words_count"] / (df["text_len_tokens"] + 1)
df["cut_words_ratio"]  = df["cut_words_count"]  / (df["text_len_tokens"] + 1)
df["hold_words_ratio"] = df["hold_words_count"] / (df["text_len_tokens"] + 1)

# контрастный признак: насколько текст "ястребинее", чем "голубинее"
df["hike_minus_cut_ratio"] = df["hike_words_ratio"] - df["cut_words_ratio"]

слов для hike (log-odds): 50
слов для cut  (log-odds): 50
слов для hold (freq):     50


In [23]:
df = df.dropna(subset='decision_future')

In [25]:
# сохраняем df для дальнейшей работы моделей

df.to_csv('data_ml.csv', index=False)